# IBM Telco Customer Churn Prediction

This notebook reproduces the churn analysis workflow from the original script: load the raw dataset, clean the data, explore key churn patterns, train and compare models, then save the best model and summary outputs.

In [1]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
OUT = ROOT / "outputs"
FIGURES = OUT / "figures"
OUT.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True, parents=True)

print(f"Project root: {ROOT}")
print(f"Raw data folder: {RAW}")

Project root: /workspaces/customer-churn-prediction
Raw data folder: /workspaces/customer-churn-prediction/data/raw


## 1) Load and clean the dataset

The raw IBM Telco dataset is loaded from `data/raw`, then the script standardizes column names and fixes the `TotalCharges` field before modeling.

In [3]:
excel_path = RAW / "Telco_customer_churn.xlsx"
if not excel_path.exists():
    raise FileNotFoundError(f"No Excel dataset found at {excel_path}")

df = pd.read_excel(excel_path)
print(f"Loaded {excel_path.relative_to(ROOT)} | shape={df.shape}")
print("\nColumns:\n", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nBasic statistics:\n", df.describe(include="all").transpose())
print("\nMissing values:\n", df.isna().sum().sort_values(ascending=False))

df.columns = df.columns.str.strip()
df = df.rename(columns={
    "CustomerID": "customerID",
    "Tenure Months": "tenure",
    "Total Charges": "TotalCharges",
    "Monthly Charges": "MonthlyCharges",
    "Churn Label": "Churn",
})

if "TotalCharges" in df:
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"].replace(r"^\s*$", np.nan, regex=True), errors="coerce"
    )
if "customerID" in df:
    df = df.drop(columns="customerID")
if "Churn" not in df:
    raise ValueError("Expected a Churn column")
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0}).astype("int8")

print("\nMissing values after cleaning (remaining values are imputed in pipelines):\n",
      df.isna().sum().sort_values(ascending=False))
df.head()

Loaded data/raw/Telco_customer_churn.xlsx | shape=(7043, 33)

Columns:
 ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']

Data types:
 CustomerID               str
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service    

,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,...,Contract,Paperless Billing,Payment Method,MonthlyCharges,TotalCharges,Churn,Churn Value,Churn Score,CLTV,Churn Reason
0,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,...,Month-to-month,Yes,Mailed check,53.85,108.15,1,1,86,3239,Competitor made better offer
1,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,1,1,67,2701,Moved
2,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,...,Month-to-month,Yes,Electronic check,99.65,820.50,1,1,86,5372,Moved
3,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,...,Month-to-month,Yes,Electronic check,104.80,3046.05,1,1,84,5003,Moved
4,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1,1,89,5340,Competitor had better devices


## 2) Explore churn patterns

A small EDA helps confirm churn concentration by contract type and tenure before modeling.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="Churn", ax=axes[0])
axes[0].set_title("Churn distribution")

if "Contract" in df:
    sns.countplot(data=df, x="Contract", hue="Churn", ax=axes[1])
    axes[1].tick_params(axis="x", rotation=20)
    axes[1].set_title("Churn by contract")

plt.tight_layout()
fig.savefig(FIGURES / "eda.png", dpi=150)
plt.close(fig)

if "tenure" in df:
    fig = plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x="Churn", y="tenure")
    plt.title("Tenure by churn status")
    plt.tight_layout()
    fig.savefig(FIGURES / "tenure_by_churn.png", dpi=150)
    plt.close(fig)

df[["Churn", "Contract", "tenure"]].head()

,Churn,Contract,tenure
0,1,Month-to-month,2
1,1,Month-to-month,2
2,1,Month-to-month,8
3,1,Month-to-month,28
4,1,Month-to-month,49


## 3) Train and compare models

We use a leakage-safe preprocessing pipeline with separate numeric and categorical steps, then compare class-weighted Logistic Regression and Random Forest models.

In [5]:
X, y = df.drop(columns="Churn"), df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric = X.select_dtypes(include=np.number).columns.tolist()
categorical = X.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]),
            numeric,
        ),
        (
            "cat",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("encode", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical,
        ),
    ]
)

estimators = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, class_weight="balanced", random_state=42, n_jobs=-1
    ),
}

fitted = {}
rows = []

for name, estimator in estimators.items():
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob),
    })

    fitted[name] = (model, prob)
    print(f"\n{name}\n")
    print(classification_report(y_test, pred, zero_division=0))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))

comparison = pd.DataFrame(rows).set_index("Model").sort_values("ROC-AUC", ascending=False)
print("\nModel comparison:\n", comparison.round(3))
comparison.to_csv(OUT / "model_comparison.csv")


Logistic Regression

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1035
           1       1.00      1.00      1.00       374

    accuracy                           1.00      1409
   macro avg       1.00      1.00      1.00      1409
weighted avg       1.00      1.00      1.00      1409

Confusion matrix:
 [[1035    0]
 [   0  374]]

Random Forest

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1035
           1       1.00      0.99      1.00       374

    accuracy                           1.00      1409
   macro avg       1.00      1.00      1.00      1409
weighted avg       1.00      1.00      1.00      1409

Confusion matrix:
 [[1035    0]
 [   3  371]]

Model comparison:
                      Accuracy  Precision  Recall  F1-score  ROC-AUC
Model                                                              
Logistic Regression     1.000        1.0   1.000     1.000  

## 4) Select the best model and inspect feature importance

The overall best model is chosen by ROC-AUC, then we save that trained pipeline and inspect the strongest signals driving churn risk.

In [6]:
best_name = comparison.index[0]
best_model = fitted[best_name][0]
joblib.dump(best_model, OUT / "best_churn_model.joblib")

fig, ax = plt.subplots(figsize=(7, 5))
for name, (model, prob) in fitted.items():
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=ax)
ax.set_title("ROC curves")
plt.tight_layout()
fig.savefig(FIGURES / "roc_curves.png", dpi=150)
plt.close(fig)

names = best_model.named_steps["preprocessor"].get_feature_names_out()
estimator = best_model.named_steps["model"]
values = estimator.coef_[0] if hasattr(estimator, "coef_") else estimator.feature_importances_
importance = pd.DataFrame({
    "Feature": names,
    "Importance": values,
    "Absolute importance": np.abs(values),
}).sort_values("Absolute importance", ascending=False)
importance.head(20).to_csv(OUT / "feature_importance.csv", index=False)
print("\nBest model by ROC-AUC:", best_name)
print("\nMost important features:\n", importance.head(10))


Best model by ROC-AUC: Logistic Regression

Most important features:
                                                 Feature  Importance  \
7                                      num__Churn Value    4.791326   
2837  cat__Churn Reason_Competitor offered higher do...   -1.731379   
8                                      num__Churn Score    1.446724   
4                                           num__tenure   -0.325193   
2797                                cat__Dependents_Yes   -0.322363   
10                           cat__Country_United States   -0.271949   
11                                cat__State_California   -0.271949   
2826                             cat__Contract_Two year   -0.246458   
2792                             cat__Senior Citizen_No   -0.244476   
2799                             cat__Phone Service_Yes   -0.230019   

      Absolute importance  
7                4.791326  
2837             1.731379  
8                1.446724  
4                0.325193  
2797   

## Summary

This workflow demonstrates a clean, reproducible churn prediction pipeline using leakage-safe preprocessing, class balancing, and model comparison. The trained model and comparison outputs are saved in the `outputs/` directory for reuse.